# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a Croissant-structured biomedical dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It follows best practices for referencing dataset entities via their `@id` values and guides you from data download to basic exploratory data analysis.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and publically accessible at the following URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL (from FAIR^2 repository)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a DatasetMetadata instance

print(f"{metadata.name}: {metadata.description}\n")
print(f"Croissant version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record set @ids and their fields
record_sets = metadata.record_sets

if not record_sets:
    print("No record sets found in metadata. Attempting fallback to FileObjects for tabular data...")
    # Fallback: Try to access Dataset 'distribution' for possible record set
    if hasattr(metadata, 'distributions') and metadata.distributions:
        for distribution in metadata.distributions:
            print(f"Distribution @id: {distribution.id if hasattr(distribution, 'id') else getattr(distribution, '@id', None)}")
            print(f"\tName: {getattr(distribution, 'name', None)}")
            if hasattr(distribution, 'file_objects') and distribution.file_objects:
                for file_object in distribution.file_objects:
                    print(f"\tFileObject @id: {file_object.id}")
            print()
    else:
        print("Did not find any Data Distributions or FileObjects either.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field @id: {f.id} | Name: {f.name} | Data type: {f.data_type}")
        print()

### Show a preview of records in a selected RecordSet

Pick one available record set (using its `@id`) for preview. If none was displayed above, check documentation for RecordSet IDs and choose the primary one.

In [ ]:
# Select the main record set's @id for further data extraction
# Assign the @id as a variable. For demonstration, we use the most likely structure:
# The actual @id may vary; please adjust after running the previous cell.

# If record_sets was empty, you must specify the correct record set found in `dataset.metadata`.
record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # Example: Data Distribution ID
# If you saw valid RecordSet @ids in the output above, replace this value with one of them.

# Preview a few records
try:
    preview = list(dataset.records(record_set=record_set_id))
    print(f"Found {len(preview)} records in RecordSet '@id': {record_set_id}\n")
    print(json.dumps(preview[:2], indent=2))  # Show two example records
except Exception as e:
    print(f"Failed to read records for RecordSet '@id' {record_set_id}\nError: {e}")

## 3. Data Extraction
Load all data from each available RecordSet into DataFrames for analysis. Use only `@id`s to reference record sets.

If you have more than one meaningful record set, include all their `@id`s in the list below. Otherwise, just use the primary one.

In [ ]:
record_sets_ids = [
    'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
]  # Expand this list with other record set @ids if applicable.

dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"RecordSet {rs_id}: DataFrame shape = {dataframes[rs_id].shape}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load RecordSet @id {rs_id}: {e}")

# Show the first rows of the primary DataFrame
main_rs_id = record_sets_ids[0]
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's now select a numeric field for analysis. We will demonstrate filtering, normalization, and simple grouping operations.

> **NOTE:**
> - You *must* reference fields by their actual DataFrame column name, which often matches the dataset Field's `@id` (unless an explicit column name was set). Use `.columns` from above to select.
> - Adjust `numeric_field_id` and `group_field_id` to fit this dataset.

In [ ]:
# Identify a numeric field and a grouping field using @id from the overview (see previous columns)
# Example field names below are illustrative; adjust as needed to your DataFrame columns
# E.g., numeric_field_id = 'cr:mw'  # if column is 'cr:mw'
df = dataframes[main_rs_id]

# Set to a numeric column from the DataFrame
if len(df.columns) > 0:
    print(f"Available columns: {df.columns.tolist()}")
else:
    print("No data loaded. Check RecordSet @id and schema.")

# Select actual field IDs; example guesses below:
numeric_field_id = None
possible_numeric = ['cr:mw', 'cr:coverage', 'cr:unique_peptides', 'cr:abundance']
for col in df.columns:
    if col in possible_numeric:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback: pick first float/int column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

group_field_id = None
possible_group = ['cr:description', 'cr:accession']
for col in df.columns:
    if col in possible_group:
        group_field_id = col
        break

print(f"Using numeric field '@id': {numeric_field_id}")
print(f"Using group field '@id': {group_field_id}")

if numeric_field_id is not None:
    # Remove missing values
    col_ser = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[col_ser > col_ser.mean()]  # Example: filter above mean
    print(f"Filtered records where {numeric_field_id} > mean: {len(filtered_df)} rows")
    # Normalize field
    filtered_df[numeric_field_id + "_normalized"] = (col_ser - col_ser.mean()) / col_ser.std()
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (first 5):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found.")

## 5. Visualization

Visualize the distribution of the selected numeric field and relationships (e.g., histogram, boxplot, or scatter).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No valid numeric field to visualize.")

## 6. Conclusion
In this notebook, you successfully loaded and explored the FAIR^2 mass spectrometry extracellular vesicle dataset via its Croissant schema using `mlcroissant`:
- Inspected available record sets, fields, and sample data.
- Loaded the primary record set into a DataFrame and performed basic filtering, normalization, grouping, and visualizations.

For deeper analysis, refer to the [schema fields' `@id` documentation](https://mlcommons.org/croissant/) and examine additional attributes/record sets as needed.